# 02 Shapiro-Wilk Check

各グループの重心座標が正規分布に従っているか確認するNotebook。

- 入力: `data/processed/{DATASET_NAME}/team_centroids_by_sequence.csv`
- 必須列: `group`, `mean_centroid_x`, `mean_centroid_y`
- 有意水準: `alpha = 0.05`
- nが3未満のグループはShapiro-Wilk検定を実行しない
- nが20未満のグループは `note` に注意を出す

In [ ]:
from pathlib import Path

import pandas as pd
from scipy.stats import shapiro

PROJECT_ROOT = Path.cwd().parent.parent if Path.cwd().name in ('重心', 'effective_area') else Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
MATCH_DATE = '2023-11-25'
MATCH_ID = '117093'
MATCH_LABEL = 'tsukuba_vs_tsukuba-b'
DATASET_NAME = f'{MATCH_DATE}_{MATCH_ID}_{MATCH_LABEL}'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed' / DATASET_NAME

INPUT_PATH = PROCESSED_DIR / 'team_centroids_by_sequence.csv'

ALPHA = 0.05
MIN_SHAPIRO_N = 3
SHAPIRO_WARNING_N = 20
REQUIRED_COLUMNS = ['group', 'mean_centroid_x', 'mean_centroid_y']


In [ ]:
def load_centroid_sequences(path):
    if not path.exists():
        raise FileNotFoundError(f'重心分析用CSVが見つかりません: {path}')
    df = pd.read_csv(path)
    missing = [col for col in REQUIRED_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f'必須列が不足しています: {missing}')
    return df


def shapiro_wilk_by_group(df, value_col, group_col='group', alpha=ALPHA):
    rows = []
    for group, sub in df.groupby(group_col):
        values = sub[value_col].dropna().to_numpy()
        n = len(values)
        checked = n >= MIN_SHAPIRO_N
        if not checked:
            statistic, p_value = float('nan'), float('nan')
            note = f'n={n} のためShapiro-Wilk検定は実行不可。最低 {MIN_SHAPIRO_N} 件必要。'
        else:
            statistic, p_value = shapiro(values)
            note = 'nが少ないため解釈注意。' if n < SHAPIRO_WARNING_N else ''
        rows.append({
            'test': 'shapiro_wilk',
            'variable': value_col,
            'group': group,
            'n': n,
            'statistic': statistic,
            'p_value': p_value,
            'alpha': alpha,
            'passed': bool(pd.notna(p_value) and p_value > alpha),
            'checked': checked,
            'note': note,
        })
    return pd.DataFrame(rows)

In [ ]:
df = load_centroid_sequences(INPUT_PATH)
df.head()

## x 方向

In [ ]:
shapiro_x = shapiro_wilk_by_group(df, 'mean_centroid_x')
shapiro_x

## y 方向

In [ ]:
shapiro_y = shapiro_wilk_by_group(df, 'mean_centroid_y')
shapiro_y